In [1]:
import arviz as az
import IPython
from meridian import constants
from meridian.analysis import analyzer
from meridian.analysis import formatter
from meridian.analysis import optimizer
from meridian.analysis import summarizer
from meridian.analysis import visualizer
from meridian.data import data_frame_input_data_builder
from meridian.data import test_utils
from meridian.model import model
from meridian.model import prior_distribution
from meridian.model import spec
import numpy as np
import pandas as pd
# check if GPU is available
from psutil import virtual_memory
import tensorflow as tf
import tensorflow_probability as tfp

ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
print(
    'Num GPUs Available: ',
    len(tf.config.experimental.list_physical_devices('GPU')),
)
print(
    'Num CPUs Available: ',
    len(tf.config.experimental.list_physical_devices('CPU')),
)

Your runtime has 25.8 gigabytes of available RAM

Num GPUs Available:  0
Num CPUs Available:  1


In [2]:
test_dir = "/Users/mariappan.subramanian/Library/CloudStorage/OneDrive-TheTradeDesk/MMM/BudgetOptimizer/trash"

In [3]:
# # 1. load input data
# df = pd.read_csv(
#     "https://raw.githubusercontent.com/google/meridian/refs/heads/main/meridian/data/simulated_data/csv/geo_media_rf.csv"
# )
# # 2. Create a DataFrameInputDataBuilder instance
# builder = data_frame_input_data_builder.DataFrameInputDataBuilder(
#     kpi_type='non_revenue'
# )
# builder = (
#     builder.with_kpi(df, kpi_col="conversions")
#     .with_revenue_per_kpi(df, revenue_per_kpi_col="revenue_per_conversion")
#     .with_population(df)
#     .with_controls(
#         df,
#         control_cols=[
#             "sentiment_score_control",
#             "competitor_activity_score_control",
#         ],
#     )
# )

# channels = ["Channel0", "Channel1", "Channel2"]
# builder = builder.with_media(
#     df,
#     media_cols=[f"{channel}_impression" for channel in channels],
#     media_spend_cols=[f"{channel}_spend" for channel in channels],
#     media_channels=channels,
# ).with_reach(
#     df,
#     reach_cols=["Channel3_reach"],
#     frequency_cols=["Channel3_frequency"],
#     rf_spend_cols=["Channel3_spend"],
#     rf_channels=["Channel3"],
# )

# data = builder.build()

# # Configure the model
# roi_rf_mu = 0.2  # Mu for ROI prior for each RF channel.
# roi_rf_sigma = 0.9  # Sigma for ROI prior for each RF channel.
# prior = prior_distribution.PriorDistribution(
#     roi_rf=tfp.distributions.LogNormal(
#         roi_rf_mu, roi_rf_sigma, name=constants.ROI_RF
#     )
# )
# model_spec = spec.ModelSpec(prior=prior)

# mmm = model.Meridian(input_data=data, model_spec=model_spec)

1. load a sample model object

In [4]:
demo_model_path = '/Users/mariappan.subramanian/Documents/repo/forked/meridian/demo/saved_models'
demo_model_file = f"{demo_model_path}/demo_model_geo_all_channels.pkl"

mmm = model.load_mmm(demo_model_file)

In [5]:
%%time
budget_optimizer = optimizer.BudgetOptimizer(mmm)
optimization_results = budget_optimizer.optimize()

I0000 00:00:1756497168.217369 2322363 service.cc:148] XLA service 0x10e9e53d0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1756497168.217403 2322363 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1756497168.225059 2322363 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-08-29 14:52:48.774370: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


CPU times: user 3min 9s, sys: 47.8 s, total: 3min 57s
Wall time: 1min 35s


In [6]:
optimization_results.plot_spend_delta()

alt.LayerChart(...)

In [7]:
optimization_results.plot_incremental_outcome_delta()

alt.LayerChart(...)

In [8]:
optimization_results.plot_budget_allocation()

alt.Chart(...)

In [9]:
optimization_results.plot_response_curves()

alt.FacetChart(...)

In [10]:
optimization_results.nonoptimized_data.spend.values

array([40400000, 27600000, 23300000, 19600000])

In [11]:
optimization_results.optimized_data.spend.values

array([28300000, 27000000, 30100000, 25500000])

In [12]:
optimization_results.optimized_data

<xarray.Dataset> Size: 512B
Dimensions:              (channel: 4, metric: 4)
Coordinates:
  * channel              (channel) object 32B 'Channel0' ... 'Channel3'
  * metric               (metric) <U6 96B 'mean' 'median' 'ci_lo' 'ci_hi'
Data variables:
    spend                (channel) int64 32B 28300000 27000000 30100000 25500000
    pct_of_spend         (channel) float64 32B 0.2552 0.2435 0.2714 0.2299
    incremental_outcome  (channel, metric) float32 64B 5.405e+07 ... 1.493e+08
    effectiveness        (channel, metric) float32 64B 0.02124 ... 0.06886
    roi                  (channel, metric) float32 64B 1.91 1.824 ... 5.855
    mroi                 (channel, metric) float32 64B 1.254 1.246 ... 5.856
    cpik                 (channel, metric) float32 64B 0.5513 0.5483 ... 0.1966
Attributes:
    start_date:                 2021-01-25
    end_date:                   2024-01-15
    budget:                     110900000.0
    profit:                     238578660.0
    total_incremental_outcome:  349478660.0
    total_roi:                  3.1512954
    total_cpik:                 0.31812632
    is_revenue_kpi:             True
    confidence_level:           0.9
    use_historical_budget:      True
    fixed_budget:               True

manual run

In [4]:
from collections.abc import Mapping, Sequence
import dataclasses
import functools
import math
import os
from typing import Any, TypeAlias
import warnings

import altair as alt
import jinja2
from meridian import constants as c
from meridian.analysis import analyzer
from meridian.analysis import formatter
from meridian.analysis import summary_text
from meridian.data import time_coordinates as tc
from meridian.model import model
import numpy as np
import pandas as pd
import tensorflow as tf
import xarray as xr

from meridian.analysis.optimizer import _SpendConstraint, OptimizationGrid, _validate_budget

In [5]:
# initialize
self = optimizer.BudgetOptimizer(mmm)

In [6]:
self

In [7]:
new_data: analyzer.DataTensors | None = None
use_posterior: bool = True
selected_times: tuple[str | None, str | None] | None = None
start_date: tc.Date = None
end_date: tc.Date = None
fixed_budget: bool = True
budget: float | None = None
pct_of_spend: Sequence[float] | None = None
spend_constraint_lower: _SpendConstraint | None = None
spend_constraint_upper: _SpendConstraint | None = None
target_roi: float | None = None
target_mroi: float | None = None
gtol: float = 0.0001
use_optimal_frequency: bool = True
use_kpi: bool = False
confidence_level: float = c.DEFAULT_CONFIDENCE_LEVEL
batch_size: int = c.DEFAULT_BATCH_SIZE
optimization_grid: OptimizationGrid | None = None

In [8]:
if selected_times is not None:
  warnings.warn(
      '`selected_times` is deprecated. Please use `start_date` and'
      ' `end_date` instead.',
      DeprecationWarning,
      stacklevel=2,
  )
  deprecated_start_date, deprecated_end_date = selected_times
  start_date = start_date or deprecated_start_date
  end_date = end_date or deprecated_end_date

_validate_budget(
    fixed_budget=fixed_budget,
    budget=budget,
    target_roi=target_roi,
    target_mroi=target_mroi,
)

In [9]:
spend_constraint_default = (
    c.SPEND_CONSTRAINT_DEFAULT_FIXED_BUDGET
    if fixed_budget
    else c.SPEND_CONSTRAINT_DEFAULT_FLEXIBLE_BUDGET
)

if spend_constraint_lower is None:
  spend_constraint_lower = spend_constraint_default
if spend_constraint_upper is None:
  spend_constraint_upper = spend_constraint_default


In [10]:
spend_constraint_default, spend_constraint_lower, spend_constraint_upper

(0.3, 0.3, 0.3)

In [11]:
use_grid_arg = optimization_grid is not None and self._validate_grid(
    new_data=new_data,
    use_posterior=use_posterior,
    start_date=start_date,
    end_date=end_date,
    budget=budget,
    pct_of_spend=pct_of_spend,
    spend_constraint_lower=spend_constraint_lower,
    spend_constraint_upper=spend_constraint_upper,
    gtol=gtol,
    use_optimal_frequency=use_optimal_frequency,
    use_kpi=use_kpi,
    optimization_grid=optimization_grid,
)

In [12]:
use_grid_arg

False

In [13]:
if optimization_grid is None or not use_grid_arg:
  optimization_grid = self.create_optimization_grid(
      new_data=new_data,
      start_date=start_date,
      end_date=end_date,
      budget=budget,
      pct_of_spend=pct_of_spend,
      spend_constraint_lower=spend_constraint_lower,
      spend_constraint_upper=spend_constraint_upper,
      gtol=gtol,
      use_posterior=use_posterior,
      use_kpi=use_kpi,
      use_optimal_frequency=use_optimal_frequency,
      batch_size=batch_size,
  )

I0000 00:00:1756490321.706679 2058202 service.cc:148] XLA service 0x104cf4d70 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1756490321.706920 2058202 service.cc:156]   StreamExecutor device (0): Host, Default Version
I0000 00:00:1756490321.799894 2058202 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-08-29 12:58:42.347144: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


In [16]:
mean_opt_results = optimization_grid.grid_dataset.mean(dim="grid_spend_index")

In [18]:
mean_opt_results.to_pandas()

,spend_grid,incremental_outcome_grid
channel,,
Channel0,40400000.0,6.693500e+07
Channel1,27600000.0,6.857640e+07
Channel2,23300000.0,7.662348e+07
Channel3,19600000.0,1.071444e+08


In [ ]:
optimization_results.plot_spend_delta()

In [23]:
# inside create_optimization_grid
# optimization_grid = self.create_optimization_grid(
#     new_data=new_data,
#     start_date=start_date,
#     end_date=end_date,
#     budget=budget,
#     pct_of_spend=pct_of_spend,
#     spend_constraint_lower=spend_constraint_lower,
#     spend_constraint_upper=spend_constraint_upper,
#     gtol=gtol,
#     use_posterior=use_posterior,
#     use_kpi=use_kpi,
#     use_optimal_frequency=use_optimal_frequency,
#     batch_size=batch_size,
# )
batch_size: int = c.DEFAULT_BATCH_SIZE  # default 100
if new_data is None:
  new_data = analyzer.DataTensors()


In [31]:
required_tensors = c.PERFORMANCE_DATA + (c.TIME,)
required_tensors = c.PERFORMANCE_DATA + (c.TIME,)
filled_data = new_data.validate_and_fill_missing_data(
    required_tensors_names=required_tensors, meridian=self._meridian
)

In [ ]:
#

filled_data.media

<tf.Tensor: shape=(20, 156, 3), dtype=float32, numpy=
array([[[1392518.,    3733.,  670235.],
        [ 937228.,  722210.,  745025.],
        [1286569.,  329778.,  786262.],
        ...,
        [1211774., 1173873.,       0.],
        [ 836566.,  305098.,  407998.],
        [1269842., 1263794.,  509309.]],

       [[3032312., 1231404.,  763501.],
        [2785805., 1548845., 1290322.],
        [2811866., 1705897.,  993765.],
        ...,
        [2274402., 2511346.,   14731.],
        [ 786126.,       0., 1411209.],
        [2067059., 2773346., 1458800.]],

       [[ 593030.,  438756.,  137622.],
        [ 534426.,  360286.,  118274.],
        [ 630650.,  424657.,  326189.],
        ...,
        [ 487092.,  598387.,  160095.],
        [ 292763.,  272152.,  237176.],
        [ 495482.,  535492.,  181453.]],

       ...,

       [[ 352217.,   31117.,   21118.],
        [ 313975.,  143726.,  252434.],
        [ 269650.,  244641.,   57590.],
        ...,
        [ 249370.,  264544.,   2174